In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Module Import

In [ ]:
!pip install pandas ftfy google-generativeai
!pip install torch

In [ ]:
import pandas as pd
import numpy as np
from  matplotlib import pylab as plt

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, LlamaConfig, LlamaForCausalLM

from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# This should return True if your NVIDIA GPU is recognized
print(torch.cuda.is_available())
# Optional: Check the name of your GPU
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

True
Tesla T4
Using device: cuda


# Data Import

- **Size**: 32,583 rows.
- **Columns**: Two columns: `sentiment` (positive, neutral, negative) and `text` (the actual news snippet or tweet).
- **Domain**: It is clearly a **Financial News/Market Sentiment** dataset. Example texts include RBI announcements, Dow Jones movements, and oil-rig counts. There are a few missing values in the `text` column (14 nulls) that we will need to clean.

In [ ]:
# Load the dataset to inspect its structure and content
try:
    dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/Finance News Sentiments (LLM/dataset.csv")

    # Get basic info
    info_buf = []
    import io
    buf = io.StringIO()
    dataset.info(buf=buf)
    info_str = buf.getvalue()

    # Get the first few rows
    head_str = dataset.head().to_string()

    print("--- INFO ---")
    print(info_str)
    print("\n--- HEAD ---")
    print(head_str)
except Exception as e:
    print(f"Error loading dataset: {e}")

--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32583 entries, 0 to 32582
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  32583 non-null  object
 1   text       32569 non-null  object
dtypes: object(2)
memory usage: 509.2+ KB


--- HEAD ---
  sentiment                                                                                                                  text
0  positive  All banks, lending institutions may allow a three-month moratorium on all loans, says RBI chief Shaktikanta Das ami…
1   neutral                                                                                                 Not so fast, Drake...
2  positive            FNF - ong  19.43. Trailing Stop  21.04 from 6 prior Stops of 19.71, 19.52, 18.70, 18.65, 18.07 and 17.87 -
3  positive                                      Dow opens down almost 500 points after China hits back with new round of tariffs
4  positive     

# Step 1: Decide on the Pre-training Strategy

You have two main paths for this dataset:

1. **Continued Causal Pre-training (Language Modeling):** You feed only the `text` column into the model so it learns the *vocabulary and structure* of financial news (e.g., learning that "Dow opens down" is a common sequence).
2. **Instruction Pre-training (Formatting):** You combine both columns into a single string to teach the model to associate specific financial texts with their market sentiment.

For domain adaptation, let's use the **Instruction formatting** approach so the model learns to understand financial text and output sentiment.

# Step 2: Prepare the Dataset in Python

Before feeding this to our initialized Transformer, we need to clean the nulls, format the text, and tokenize it.

Here is the exact code to prepare dataset.csv for your custom architecture:

In [ ]:
# 1. Load and Clean the Data
df = dataset.copy()
df = df.dropna(subset=['text']) # Drop the 14 missing text rows

# 2. Format into a Causal Language Modeling structure
# We want the model to read the text and predict the sentiment.
def format_row(row):
    return f"[FIN-TEXT] {row['text']} [FIN-SENTIMENT] {row['sentiment']} <|endoftext|>"

df['formatted_text'] = df.apply(format_row, axis=1)

# 3. Convert to Hugging Face Dataset format
hf_dataset = Dataset.from_pandas(df[['formatted_text']].head(len(df) - 200))

# 4. Tokenization
# Assuming you are using the LLaMA/Mistral tokenizer we discussed earlier
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-step-50K-105b")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples["formatted_text"],
        truncation=True,
        padding="max_length",
        max_length=128 # Financial tweets/news are usually short
    )

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
print(tokenized_datasets[0])

Map:   0%|          | 0/32369 [00:00<?, ? examples/s]

{'formatted_text': '[FIN-TEXT] All banks, lending institutions may allow a three-month moratorium on all loans, says RBI chief Shaktikanta Das ami… [FIN-SENTIMENT] positive <|endoftext|>', '__index_level_0__': 0, 'input_ids': [1, 518, 29943, 1177, 29899, 16975, 29962, 2178, 24388, 29892, 301, 2548, 21561, 1122, 2758, 263, 2211, 29899, 10874, 3036, 1061, 1974, 373, 599, 658, 550, 29892, 4083, 390, 12809, 9087, 1383, 5867, 638, 6949, 3713, 18926, 30098, 518, 29943, 1177, 29899, 29903, 3919, 7833, 3919, 29962, 6374, 529, 29989, 355, 974, 726, 29989, 29958, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Step 3: Connecting it to Your Custom Architecture

In your original prompt, you mentioned: *"You don't write the Transformer code; you just initialize its architecture and train it..."*

Now that tokenized_datasets is ready, you would pass it directly into the Hugging Face Trainer along with the randomly initialized model we created in the previous step:

In [ ]:
print("Initializing Micro-LLaMA model...")

# 1. Define a heavily scaled-down LLaMA architecture
config = LlamaConfig(
    vocab_size=tokenizer.vocab_size, # Match your tokenizer
    hidden_size=512,         # Standard is 2048+; we use 512
    intermediate_size=1024,  # Standard is 5632+; we use 1024
    num_hidden_layers=4,     # Standard is 22+; we use 4 layers
    num_attention_heads=8,   # Standard is 32; we use 8
    max_position_embeddings=512, # Max context window
)

# 2. Initialize random weights
model = LlamaForCausalLM(config)

print(f"Model ready with {model.num_parameters() / 1e6:.2f} Million parameters!")


# --- PHASE 4: THE TRAINING LOOP ---
print("Setting up Trainer...")
# 1. Set up the Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, # Make sure your tokenizer cell from earlier was run!
    mlm=False
)

# 2. Define Training Arguments (removed the problematic overwrite_output_dir)
training_args = TrainingArguments(
    output_dir="./financial_sentiment_model",
    num_train_epochs=3,
    per_device_train_batch_size=4, # Lowered to 4 to prevent Out of Memory (OOM) errors
    save_steps=500,
    logging_steps=50,
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets, # Make sure your dataset cell from earlier was run!
    data_collator=data_collator,
)

# 4. Blast off!
print("Starting training...")
trainer.train()

Initializing Micro-LLaMA model...
Model ready with 43.26 Million parameters!
Setting up Trainer...
Starting training...


Step,Training Loss
50,8.126299
100,5.596895
150,4.549725
200,4.319456
250,4.114171
300,4.105911
350,3.964040
400,4.112497
450,4.093737
500,4.047820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24279, training_loss=2.847704177014915, metrics={'train_runtime': 1442.0839, 'train_samples_per_second': 67.338, 'train_steps_per_second': 16.836, 'total_flos': 2004241346592768.0, 'train_loss': 2.847704177014915, 'epoch': 3.0})

# Step 4: The Evaluation

In [ ]:
print("Loading model and tokenizer...")
# 1. Load the tokenizer and your saved model
#tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-step-50K-105b")
#model_path = "./financial_sentiment_model/checkpoint-500" # Update this if your final checkpoint number is different
#model = AutoModelForCausalLM.from_pretrained(model_path)
model.eval() # Put the model in evaluation mode

# Move to GPU to make predictions fast
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Prepare the Test Data
# We are taking the last 200 rows of the dataset to act as our "unseen" test set
#df = pd.read_csv('dataset.csv').dropna(subset=['text'])
test_df = df.tail(200)

true_labels = []
predictions = []

# 3. The Inference Loop
print("Running evaluation on test data...")
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    text = row['text']
    actual_sentiment = row['sentiment'].strip().lower()

    # Format the prompt, leaving the end blank for the model to fill in
    prompt = f"[FIN-TEXT] {text} [FIN-SENTIMENT]"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate the prediction
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3, # We only need it to output 1 word (positive/neutral/negative)
            temperature=0.1,  # Low temp keeps it highly deterministic
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode the AI's output
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the word the model generated after the prompt tag
    try:
        predicted_word = full_output.split("[FIN-SENTIMENT]")[1].strip().lower()
        # Clean up any trailing punctuation
        predicted_word = "".join(c for c in predicted_word if c.isalpha())
    except IndexError:
        predicted_word = "format_error" # Catches if the model hallucinated badly

    true_labels.append(actual_sentiment)
    predictions.append(predicted_word)

# 4. The Final Report
print("\n--- Evaluation Results ---")
print(f"Overall Accuracy: {accuracy_score(true_labels, predictions) * 100:.2f}%\n")
print("Detailed Report (Precision, Recall, F1-Score):")
# We suppress warnings here in case the model never guessed a specific class
print(classification_report(true_labels, predictions, zero_division=0))

Loading model and tokenizer...
Running evaluation on test data...


100%|██████████| 200/200 [00:04<00:00, 42.15it/s]


--- Evaluation Results ---
Overall Accuracy: 65.00%

Detailed Report (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

    negative       0.61      0.62      0.61        60
     neutral       0.72      0.80      0.76        74
    positive       0.60      0.52      0.55        66

    accuracy                           0.65       200
   macro avg       0.64      0.64      0.64       200
weighted avg       0.65      0.65      0.65       200



## Here is the diagnosis of your model's current "brain" based on that detailed report:

1. It is highly confident about "Neutral" news
Your model scored best on the neutral class (F1-Score: 0.76, Recall: 0.80). This makes perfect sense for the financial domain. Most market news is written in a dry, factual tone (e.g., "Company X reports Q3 earnings on Tuesday"). The model successfully learned that if a headline doesn't explicitly use emotionally charged words, it should default to neutral.

2. It struggles to recognize "Positive" news
This is your model's biggest weakness. The recall for positive news is only 0.52. This means out of all the genuinely positive headlines in the test set, your model missed nearly half of them (likely misclassifying them as neutral). Financial "positivity" is tricky for AI. A phrase like "Unemployment drops to 4%" is fundamentally positive for the market, but the word "drops" usually implies something negative to a standard language model.

3. It is "Okay" at "Negative" news
With an F1-Score of 0.61, it does a decent job catching bad news, but it still has room for improvement. Negative headlines often use strong words ("plunges," "lawsuit," "bankruptcy"), which makes them slightly easier for the model to catch than positive ones.

## How to boost this to 80%+
If you were doing this for a real quantitative hedge fund, 65% wouldn't quite cut it for live trading. To push this model to the next level, you have three main levers to pull:

- Train for longer: You only trained for 3 epochs. Bumping the num_train_epochs to 5 or 8 gives the model more time to memorize the nuances of positive vs. negative financial jargon.

- Prompt Engineering: Our [FIN-TEXT] and [FIN-SENTIMENT] tags were very basic. You can often boost performance by changing the training formatting to be more conversational: Instruction: Classify the following financial headline as positive, negative, or neutral. Headline: {text} Answer: {sentiment}.

- Error Analysis: The most powerful data science technique is simply printing out the exact rows the model got wrong and reading them. You might find that the dataset itself has "bad" labels where humans disagreed on the sentiment!